# KELT-20b: Lightkurve Pipeline to `tess_proc`

This notebook does two things in sequence:

1. Downloads TESS SPOC light curves from MAST with `lightkurve` and visualizes a full cleaning/processing workflow.
2. Uses the processed TESS light curve to estimate a secondary eclipse depth, then runs and visualizes `pipeline.tess_proc` operations on that measurement.

Target defaults to KELT-20b (`TIC 69679391`).

In [ ]:
import importlib.util

required = ["lightkurve", "jax", "numpyro", "jaxoplanet"]
missing = [m for m in required if importlib.util.find_spec(m) is None]

if missing:
    print("Missing packages:", ", ".join(missing))
    print("Install in this environment before running the full notebook.")
    print("Example: pip install lightkurve jax numpyro jaxoplanet")
else:
    print("All required packages are importable.")

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np

import lightkurve as lk

import config
from dataio.load import load_nasa_archive_spectrum
from pipeline.tess_proc import (
    load_tess_bandpass,
    run_tess_proc,
)

warnings.filterwarnings("ignore", category=UserWarning)
plt.style.use("seaborn-v0_8-whitegrid")

PLANET = "KELT-20b"
EPHEMERIS = "Duck24"
TARGET = "TIC 69679391"
QUALITY_BITMASK = "hard"
MAX_SECTORS = None

params = config.get_params(planet=PLANET, ephemeris=EPHEMERIS)
period_day = float(params["period"])
duration_day = float(params["duration"])
epoch_bjd = float(params["epoch"])
epoch_btjd = epoch_bjd - 2457000.0

print(f"{PLANET} ({TARGET})")
print(f"period={period_day:.8f} d | duration={duration_day:.6f} d | epoch(BTJD)={epoch_btjd:.6f}")

In [ ]:
search = lk.search_lightcurve(TARGET, mission="TESS", author="SPOC")
print(search)

if len(search) == 0:
    raise RuntimeError(f"No SPOC light curves found for {TARGET}.")

exptimes = np.asarray(search.table["exptime"], dtype=float)
finite = np.isfinite(exptimes)
unique_exp, counts = np.unique(exptimes[finite], return_counts=True)
preferred_exptime = float(unique_exp[np.argmax(counts)])
selected = search[np.isclose(exptimes, preferred_exptime)]

if MAX_SECTORS is not None:
    selected = selected[: int(MAX_SECTORS)]

print(f"\nUsing cadence {preferred_exptime:.0f} s with {len(selected)} products.")
selected

In [ ]:
lcc = selected.download_all(quality_bitmask=QUALITY_BITMASK)

if lcc is None or len(lcc) == 0:
    raise RuntimeError("Download returned no light curves.")

def _int_or_default(value, default=-1):
    try:
        return int(value)
    except (TypeError, ValueError):
        return default

rows = []
for lc in lcc:
    t = np.asarray(lc.time.value, dtype=float)
    rows.append(
        (
            _int_or_default(lc.meta.get("SECTOR")),
            _int_or_default(lc.meta.get("CAMERA")),
            _int_or_default(lc.meta.get("CCD")),
            int(np.sum(np.isfinite(t))),
            float(np.nanmin(t)),
            float(np.nanmax(t)),
        )
    )

rows = sorted(rows, key=lambda x: x[0])
print(f"Downloaded {len(rows)} sectors.")
for sector, camera, ccd, npts, tmin, tmax in rows:
    print(f"Sector {sector:>2} | Cam {camera} CCD {ccd} | N={npts:>6} | BTJD {tmin:.1f} to {tmax:.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for sector, camera, ccd, npts, tmin, tmax in rows:
    ax.plot([tmin, tmax], [sector, sector], lw=5)

ax.set_title("TESS coverage for downloaded sectors")
ax.set_xlabel("Time [BTJD]")
ax.set_ylabel("Sector")
plt.show()

In [ ]:
def _safe_window_length(n_points: int, target: int = 401) -> int:
    n = max(31, min(target, n_points - 1))
    if n % 2 == 0:
        n -= 1
    return max(n, 31)

def process_lightcurve(lc: lk.LightCurve) -> dict[str, lk.LightCurve]:
    raw = lc.select_flux("pdcsap_flux")
    clean = raw.remove_nans().remove_outliers(sigma=8)

    transit_mask = clean.create_transit_mask(
        period=period_day,
        duration=duration_day,
        transit_time=epoch_btjd,
    )

    wlen = _safe_window_length(len(clean.time.value), target=401)
    flat = clean.flatten(window_length=wlen, polyorder=2, mask=transit_mask)
    norm = flat.normalize()
    binned = norm.bin(time_bin_size=(10.0 / 60.0 / 24.0))

    return {
        "raw_pdcsap": raw,
        "clean": clean,
        "flat": flat,
        "norm": norm,
        "bin10m": binned,
    }

In [ ]:
rep_idx = int(np.argmax([len(np.asarray(lc.time.value)) for lc in lcc]))
rep_lc = lcc[rep_idx]
rep_sector = rep_lc.meta.get("SECTOR")

stages = process_lightcurve(rep_lc)

fig, axes = plt.subplots(5, 1, figsize=(12, 13), sharex=True)
for ax, key in zip(
    axes,
    ["raw_pdcsap", "clean", "flat", "norm", "bin10m"],
):
    lc_stage = stages[key]
    ax.scatter(
        np.asarray(lc_stage.time.value),
        np.asarray(lc_stage.flux.value),
        s=2,
        alpha=0.5,
    )
    ax.set_ylabel(key)

axes[0].set_title(f"Lightkurve processing stages (representative sector {rep_sector})")
axes[-1].set_xlabel("Time [BTJD]")
plt.tight_layout()
plt.show()

In [ ]:
processed_list = [process_lightcurve(lc)["norm"] for lc in lcc]
processed_lcc = lk.LightCurveCollection(processed_list)
stitched = processed_lcc.stitch()

colors = plt.cm.viridis(np.linspace(0, 1, len(processed_lcc)))
fig, ax = plt.subplots(figsize=(12, 4))
for c, lc in zip(colors, processed_lcc):
    ax.scatter(np.asarray(lc.time.value), np.asarray(lc.flux.value), s=1, color=c, alpha=0.25)

stitched_bin = stitched.bin(time_bin_size=(30.0 / 60.0 / 24.0))
ax.plot(np.asarray(stitched_bin.time.value), np.asarray(stitched_bin.flux.value), "k.", ms=2, alpha=0.8)
ax.set_title("All processed sectors and stitched light curve")
ax.set_xlabel("Time [BTJD]")
ax.set_ylabel("Normalized flux")
plt.show()

In [ ]:
def phase_bin(phase: np.ndarray, flux: np.ndarray, xlim: tuple[float, float], nbins: int = 120):
    bins = np.linspace(xlim[0], xlim[1], nbins + 1)
    idx = np.digitize(phase, bins) - 1
    centers = 0.5 * (bins[:-1] + bins[1:])

    y_med = np.full(nbins, np.nan)
    y_err = np.full(nbins, np.nan)
    for i in range(nbins):
        m = idx == i
        if np.sum(m) < 5:
            continue
        y = flux[m]
        med = np.nanmedian(y)
        mad = 1.4826 * np.nanmedian(np.abs(y - med))
        y_med[i] = med
        y_err[i] = mad / np.sqrt(max(np.sum(np.isfinite(y)), 1))

    keep = np.isfinite(y_med)
    return centers[keep], y_med[keep], y_err[keep]

folded_transit = stitched.fold(period=period_day, epoch_time=epoch_btjd)
folded_eclipse = stitched.fold(period=period_day, epoch_time=epoch_btjd + 0.5 * period_day)

phase_tr = np.asarray(folded_transit.phase.value, dtype=float)
flux_tr = np.asarray(folded_transit.flux.value, dtype=float)
phase_ec = np.asarray(folded_eclipse.phase.value, dtype=float)
flux_ec = np.asarray(folded_eclipse.flux.value, dtype=float)

tr_x, tr_y, tr_e = phase_bin(phase_tr, flux_tr, xlim=(-0.08, 0.08), nbins=120)
ec_x, ec_y, ec_e = phase_bin(phase_ec, flux_ec, xlim=(-0.08, 0.08), nbins=120)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
axes[0].errorbar(tr_x, (tr_y - 1.0) * 1e6, yerr=tr_e * 1e6, fmt=".", ms=4, alpha=0.9)
axes[0].set_title("Phase-folded transit")
axes[0].set_xlabel("Phase")
axes[0].set_ylabel("Relative flux [ppm]")

axes[1].errorbar(ec_x, (ec_y - 1.0) * 1e6, yerr=ec_e * 1e6, fmt=".", ms=4, alpha=0.9)
axes[1].set_title("Phase-folded secondary eclipse")
axes[1].set_xlabel("Phase")
plt.tight_layout()
plt.show()

In [ ]:
eclipse_half_width_phase = 0.5 * duration_day / period_day
in_eclipse = np.abs(phase_ec) <= eclipse_half_width_phase
oot_eclipse = (np.abs(phase_ec) >= 2.0 * eclipse_half_width_phase) & (np.abs(phase_ec) <= 0.15)

if np.sum(in_eclipse) < 10 or np.sum(oot_eclipse) < 10:
    raise RuntimeError("Not enough points in eclipse/oot windows.")

baseline = float(np.nanmedian(flux_ec[oot_eclipse]))
depth_frac_lc = baseline - float(np.nanmedian(flux_ec[in_eclipse]))
scatter_oot = 1.4826 * float(np.nanmedian(np.abs(flux_ec[oot_eclipse] - baseline)))
depth_err_frac_lc = scatter_oot / np.sqrt(np.sum(in_eclipse))

tbl_path = Path("input/lowres_spectra/KELT_20_b_3.12080_5244_1.tbl")
wav_ang, spec, sig, _ = load_nasa_archive_spectrum(tbl_path, mode="emission")
idx_tess = int(np.argmin(np.abs(wav_ang - 8000.0)))
depth_frac_tbl = float(spec[idx_tess])
depth_err_frac_tbl = float(sig[idx_tess])

if np.isfinite(depth_frac_lc) and depth_frac_lc > 0 and np.isfinite(depth_err_frac_lc) and depth_err_frac_lc > 0:
    eclipse_depth_obs = depth_frac_lc
    eclipse_depth_err = max(depth_err_frac_lc, 3.0e-6)
    eclipse_source = "phase-folded TESS light curve"
else:
    eclipse_depth_obs = depth_frac_tbl
    eclipse_depth_err = depth_err_frac_tbl
    eclipse_source = "NASA archive .tbl fallback"

print(f"LC estimate: {depth_frac_lc*1e6:.2f} +/- {depth_err_frac_lc*1e6:.2f} ppm")
print(f"tbl fallback: {depth_frac_tbl*1e6:.2f} +/- {depth_err_frac_tbl*1e6:.2f} ppm")
print(f"Using: {eclipse_source} -> {eclipse_depth_obs*1e6:.2f} +/- {eclipse_depth_err*1e6:.2f} ppm")

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(ec_x, (ec_y - 1.0) * 1e6, yerr=ec_e * 1e6, fmt=".", ms=4, alpha=0.8)
ax.axvspan(-eclipse_half_width_phase, eclipse_half_width_phase, alpha=0.2, color="tab:red", label="in-eclipse")
ax.axhline(-eclipse_depth_obs * 1e6, ls="--", color="tab:red", label="selected depth")
ax.set_title("Eclipse window and selected depth")
ax.set_xlabel("Phase")
ax.set_ylabel("Relative flux [ppm]")
ax.legend(loc="best")
plt.show()

## Run `tess_proc` on the TESS measurement

This runs a compact NUTS inference for notebook interactivity. Increase `num_warmup` and `num_samples` for publication-quality posteriors.

In [ ]:
output_dir = Path("output") / "kelt20b_tess_proc_notebook"
output_dir.mkdir(parents=True, exist_ok=True)

mcmc, samples, output_path = run_tess_proc(
    planet=PLANET,
    ephemeris=EPHEMERIS,
    eclipse_depth_obs=float(eclipse_depth_obs),
    eclipse_depth_err=float(eclipse_depth_err),
    transit_depth_obs=None,
    transit_depth_err=None,
    bandpass_path=None,
    download_bandpass=True,
    photon_weighted=False,
    dayside_temp_bounds=(800.0, 4500.0),
    albedo_bounds=(0.0, 1.0),
    radius_ratio_prior=None,
    radius_ratio_prior_sigma=None,
    eclipse_model_sigma=5.0e-6,
    num_warmup=400,
    num_samples=800,
    num_chains=1,
    max_tree_depth=10,
    seed=42,
    output_dir=output_dir,
)

print(f"Saved outputs in: {output_path}")
print({k: np.asarray(v).shape for k, v in samples.items()})

In [ ]:
def q1684(x):
    q16, q50, q84 = np.percentile(np.asarray(x), [16.0, 50.0, 84.0])
    return float(q16), float(q50), float(q84)

summary_keys = ["eclipse_depth", "transit_depth", "dayside_temperature", "geometric_albedo", "radius_ratio"]
for key in summary_keys:
    if key not in samples:
        continue
    q16, q50, q84 = q1684(samples[key])
    scale = 1e6 if "depth" in key else 1.0
    unit = "ppm" if "depth" in key else ""
    print(f"{key:>20}: {q50*scale:.3f} (+{(q84-q50)*scale:.3f}, -{(q50-q16)*scale:.3f}) {unit}")

plot_specs = [
    ("eclipse_depth", "Eclipse depth [ppm]", 1e6),
    ("dayside_temperature", "Dayside temperature [K]", 1.0),
    ("geometric_albedo", "Geometric albedo", 1.0),
    ("radius_ratio", "Radius ratio Rp/Rs", 1.0),
]

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, (key, label, scale) in zip(axes.ravel(), plot_specs):
    x = np.asarray(samples[key], dtype=float) * scale
    ax.hist(x, bins=35, alpha=0.8)
    q16, q50, q84 = np.percentile(x, [16.0, 50.0, 84.0])
    ax.axvline(q50, color="k", lw=1.5)
    ax.axvspan(q16, q84, color="k", alpha=0.12)
    ax.set_xlabel(label)

fig.suptitle("`tess_proc` posterior summaries")
plt.tight_layout()
plt.show()

In [ ]:
wl_m, rsp, bandpass_path = load_tess_bandpass()

def planck_lambda_np(wavelength_m: np.ndarray, temperature_k: float) -> np.ndarray:
    wl = np.asarray(wavelength_m, dtype=float)
    x = (config.H_PLANCK * config.C_LIGHT) / np.clip(wl * config.K_BOLTZMANN * temperature_k, 1e-30, None)
    x = np.clip(x, 1e-12, 700.0)
    num = 2.0 * config.H_PLANCK * config.C_LIGHT**2
    den = np.clip(wl**5 * np.expm1(x), 1e-30, None)
    return num / den

weights = rsp
star_band = np.trapz(planck_lambda_np(wl_m, float(params["T_star"])) * weights, wl_m)

radius_ratio = np.asarray(samples["radius_ratio"], dtype=float)
albedo = np.asarray(samples["geometric_albedo"], dtype=float)
tday = np.asarray(samples["dayside_temperature"], dtype=float)

planet_band = np.array([np.trapz(planck_lambda_np(wl_m, t) * weights, wl_m) for t in tday])
thermal = (radius_ratio**2) * (planet_band / star_band)
a_over_rstar = (float(params["a"]) * config.AU_M) / (float(params["R_star"]) * config.R_SUN_M)
reflected = albedo * (radius_ratio / a_over_rstar) ** 2
model_depth = thermal + reflected

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(thermal * 1e6, bins=35, alpha=0.7, label="thermal")
axes[0].hist(reflected * 1e6, bins=35, alpha=0.7, label="reflected")
axes[0].hist(model_depth * 1e6, bins=35, alpha=0.7, label="total model")
axes[0].axvline(eclipse_depth_obs * 1e6, color="k", ls="--", label="selected obs")
axes[0].set_xlabel("Depth [ppm]")
axes[0].set_title("`tess_proc` components")
axes[0].legend(loc="best")

q16, q50, q84 = np.percentile(model_depth * 1e6, [16.0, 50.0, 84.0])
axes[1].errorbar(ec_x, (ec_y - 1.0) * 1e6, yerr=ec_e * 1e6, fmt=".", ms=4, alpha=0.75)
axes[1].axhline(-q50, color="tab:red", lw=2, label="model median")
axes[1].axhspan(-q84, -q16, color="tab:red", alpha=0.18, label="model 16-84%")
axes[1].set_xlabel("Phase")
axes[1].set_ylabel("Relative flux [ppm]")
axes[1].set_title("Model depth projected on folded eclipse")
axes[1].legend(loc="best")

plt.tight_layout()
plt.show()

print(f"Bandpass file: {bandpass_path}")

The notebook writes MCMC outputs to `output/kelt20b_tess_proc_notebook/`.

If you want higher-fidelity posteriors, rerun the inference cell with larger `num_warmup` and `num_samples`.